In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import pandas as pd
import numpy as np

In [10]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [11]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")


wanted_issues = [
    "Concatenate"
]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) &
    (df["Network ID"] != 11)
]

df_concat

df_concat_new =  df_concat[['Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)

df_concat_new

,Station ID,Unique Names,Native ID
0,12939,-> Fort St John Key Learning Centre,Fort St John Key Learning Centre -> 469
1,12938,-> Pine River Gas Plant,Pine River Gas Plant -> 107
2,2640,Harmac Pacific,M102038 -> 60 -> 294
3,2623,Kelowna College,M112070 -> 102
4,12531,NaN,Birchbank Golf Course -?> 97
5,12534,NaN,Burns Lake Fire Centre -?> 104
6,12535,NaN,Burns Lake Sheraton East -?> 439
7,12537,NaN,Colwood City Hall -?> 553
8,12539,NaN,Courtenay Elementary School -> 526
9,12540,NaN,Cranbrook Muriel Baxter -?> 551


In [12]:
def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_concat_new[['old_name', 'new_name', 'n_names']] = (
    df_concat_new['Unique Names'].apply(split_station_name)
)

df_concat_new = df_concat_new.drop(columns= 'Unique Names')
df_concat_new

,Station ID,Native ID,old_name,new_name,n_names
0,12939,Fort St John Key Learning Centre -> 469,,Fort St John Key Learning Centre,2.0
1,12938,Pine River Gas Plant -> 107,,Pine River Gas Plant,2.0
2,2640,M102038 -> 60 -> 294,Harmac Pacific,Harmac Pacific,1.0
3,2623,M112070 -> 102,Kelowna College,Kelowna College,1.0
4,12531,Birchbank Golf Course -?> 97,NaN,NaN,0.0
5,12534,Burns Lake Fire Centre -?> 104,NaN,NaN,0.0
6,12535,Burns Lake Sheraton East -?> 439,NaN,NaN,0.0
7,12537,Colwood City Hall -?> 553,NaN,NaN,0.0
8,12539,Courtenay Elementary School -> 526,NaN,NaN,0.0
9,12540,Cranbrook Muriel Baxter -?> 551,NaN,NaN,0.0


In [13]:
import pandas as pd
import re

def parse_native_id(s):
    if pd.isna(s):
        return pd.Series([None, None])

    s = str(s)  # <-- KEY FIX

    parts = re.split(r'\s*-?\?>\s*|\s*->\s*', s)

    old_id = parts[0].strip()
    new_id = parts[-1].strip()

    return pd.Series([old_id, new_id])

df_concat_new[['old_native_id', 'new_native_id']] = df_concat_new['Native ID'].apply(parse_native_id)

df_concat_new = df_concat_new.drop(columns= ['Native ID', 'n_names'])

df_concat_new

,Station ID,old_name,new_name,old_native_id,new_native_id
0,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469
1,12938,,Pine River Gas Plant,Pine River Gas Plant,107
2,2640,Harmac Pacific,Harmac Pacific,M102038,294
3,2623,Kelowna College,Kelowna College,M112070,102
4,12531,NaN,NaN,Birchbank Golf Course,97
5,12534,NaN,NaN,Burns Lake Fire Centre,104
6,12535,NaN,NaN,Burns Lake Sheraton East,439
7,12537,NaN,NaN,Colwood City Hall,553
8,12539,NaN,NaN,Courtenay Elementary School,526
9,12540,NaN,NaN,Cranbrook Muriel Baxter,551


In [14]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;

"""


def count_station_stats(old_id, new_id, engine):

    params = (
    old_id,          # n_old
    new_id,          # n_new
    old_id, new_id,  # n_overlap
    old_id, new_id   # n_overlap_same_datum
    )

    df = pd.read_sql(
        query,
        engine,
        params = params
    )
    return df.iloc[0]

# df_test = df_concat_new.head(3).copy()


stats = df_concat_new.apply(
    lambda r: count_station_stats(r['old_native_id'], r['new_native_id'], engine),
    axis=1
)

df_concat_new[['n_old', 'n_new', 'n_overlap', 'n_overlap_same_datum']] = stats

In [15]:
df_concat_new

,Station ID,old_name,new_name,old_native_id,new_native_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,12939,,Fort St John Key Learning Centre,Fort St John Key Learning Centre,469,2131,95286,1150,1150
1,12938,,Pine River Gas Plant,Pine River Gas Plant,107,0,634642,0,0
2,2640,Harmac Pacific,Harmac Pacific,M102038,294,0,1973695,0,0
3,2623,Kelowna College,Kelowna College,M112070,102,4289,761324,4289,294
4,12531,NaN,NaN,Birchbank Golf Course,97,0,1021361,0,0
5,12534,NaN,NaN,Burns Lake Fire Centre,104,0,1126017,0,0
6,12535,NaN,NaN,Burns Lake Sheraton East,439,1152,289657,1152,1152
7,12537,NaN,NaN,Colwood City Hall,553,1144,181217,1144,1144
8,12539,NaN,NaN,Courtenay Elementary School,526,0,347008,0,0
9,12540,NaN,NaN,Cranbrook Muriel Baxter,551,1146,184846,1146,1146


In [17]:
df_old_0 = df_concat_new.loc[df_concat_new["n_old"] == 0, ["old_native_id", "Station ID"]]
df_old_0

,old_native_id,Station ID
1,Pine River Gas Plant,12938
2,M102038,2640
4,Birchbank Golf Course,12531
5,Burns Lake Fire Centre,12534
8,Courtenay Elementary School,12539
15,Harmac Pacific,12595
28,Trail Columbia Gardens Airport,12580
34,PG Marsulex,12560
42,Prince Rupert Roosevelt Park School,12569
49,Valemount,12581


In [20]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.station_id = :station_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_old_0.iterrows():
        station_id = row['Station ID']

        exists = conn.execute(
            exists_sql,
            {"station_id": station_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_old_0)}] "
            f"station_id={station_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
# df_old_0['new_station_status'] = exists_list

[  2/11] station_id=12938           | → EXISTS
[  3/11] station_id=2640            | → EXISTS
[  5/11] station_id=12531           | → EXISTS
[  6/11] station_id=12534           | → EXISTS
[  9/11] station_id=12539           | → EXISTS
[ 16/11] station_id=12595           | → EXISTS
[ 29/11] station_id=12580           | → EXISTS
[ 35/11] station_id=12560           | → EXISTS
[ 43/11] station_id=12569           | → EXISTS
[ 50/11] station_id=12581           | → EXISTS
[ 56/11] station_id=2618            | → EXISTS


### Check if the station exist

new_native_id 452, T035, 77, E271963, E238705, 543 donot exist

In [71]:
exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_station
    WHERE native_id = :new_id
)
""")
new_ids = '469'

with engine.connect() as conn:
    exists = conn.execute(
        exists_sql,
        {"new_id": new_ids}
    ).scalar()

exists

True

In [42]:
from sqlalchemy import text

# 1️⃣ Query to select old observations that need moving
SQL_TO_MOVE = text("""
SELECT o_old.history_id AS old_hist_id,
       o_old.obs_time,
       o_old.vars_id
FROM obs_raw o_old
JOIN meta_history h_old ON o_old.history_id = h_old.history_id
JOIN meta_station s_old ON h_old.station_id = s_old.station_id
WHERE s_old.native_id = :old_id

EXCEPT

SELECT o_new.history_id AS old_hist_id,
       o_new.obs_time,
       o_new.vars_id
FROM obs_raw o_new
JOIN meta_history h_new ON o_new.history_id = h_new.history_id
JOIN meta_station s_new ON h_new.station_id = s_new.station_id
WHERE s_new.native_id = :new_id
""")

# 2️⃣ Get the new station's history_id (assume we use latest)
SQL_NEW_HISTORY = text("""
SELECT history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id = :new_id
""")

def move_station_obs_fast(old_id, new_id, engine):
    with engine.begin() as conn:
        # get new history_id (latest)
        new_hist_id = conn.execute(SQL_NEW_HISTORY, {"new_id": new_id}).scalar()
        if new_hist_id is None:
            print(f"New station '{new_id}' has no history, skipping.")
            return 0

        # update all in one query
        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    JOIN meta_station s_old ON h_old.station_id = s_old.station_id
                    WHERE o.history_id = h_old.history_id
                      AND s_old.native_id = :old_id
                      AND NOT EXISTS (
                          SELECT 1
                          FROM obs_raw o_new
                          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
                          JOIN meta_station s_new ON h_new.station_id = s_new.station_id
                          WHERE s_new.native_id = :new_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id = o.vars_id
                      )
                    RETURNING o.*
                )
                SELECT COUNT(*) FROM updated;
            """),
            {"old_id": old_id, "new_id": new_id, "new_hist_id": new_hist_id}
        ).scalar()

        print(f"Moved {n_move} rows from '{old_id}' to '{new_id}'")
        return n_move


# 4️⃣ Loop through your dataframe
results = []

for i, row in df_concat_new.iterrows():  # preview first 2 rows
    old_id = row['old_native_id']
    new_id = row['new_native_id']
    print(f"[{i+1}/{len(df_concat_new)}] Processing old='{old_id}' -> new='{new_id}'")
    n = move_station_obs_fast(old_id, new_id, engine)
    results.append(n)

# df_concat_new['n_moved'] = results
print("All done!")

[1/56] Processing old='Fort St John Key Learning Centre' -> new='469'
New station '469' has no history, skipping.
[2/56] Processing old='Pine River Gas Plant' -> new='107'
Moved 0 rows from 'Pine River Gas Plant' to '107'
[3/56] Processing old='M102038' -> new='294'
Moved 563442 rows from 'M102038' to '294'
[4/56] Processing old='M112070' -> new='102'
Moved 469068 rows from 'M112070' to '102'
[5/56] Processing old='Birchbank Golf Course' -> new='97'
Moved 0 rows from 'Birchbank Golf Course' to '97'
[6/56] Processing old='Burns Lake Fire Centre' -> new='104'
Moved 0 rows from 'Burns Lake Fire Centre' to '104'
[7/56] Processing old='Burns Lake Sheraton East' -> new='439'
Moved 0 rows from 'Burns Lake Sheraton East' to '439'
[8/56] Processing old='Colwood City Hall' -> new='553'
Moved 0 rows from 'Colwood City Hall' to '553'
[9/56] Processing old='Courtenay Elementary School' -> new='526'
Moved 0 rows from 'Courtenay Elementary School' to '526'
[10/56] Processing old='Cranbrook Muriel Bax

In [54]:
results[1]

native_id                        107
station_name    Pine River Gas Plant
Name: 0, dtype: object

### update the station name

In [51]:
new_ids = df_concat_new['new_native_id'].tolist()

results = []

for new_id in new_ids:
    query = text("""
        SELECT s.native_id, h.station_name
        FROM meta_station s
        JOIN meta_history h ON h.station_id = s.station_id
        WHERE s.native_id = :new_id
    """)
    df_temp = pd.read_sql(query, engine, params={"new_id": new_id})
    
    if not df_temp.empty:
        results.append(df_temp.iloc[0])  # take first row (should be one per ID)
    else:
        # If the ID is not found, we can append a row with NaN for station_name
        results.append({"native_id": new_id, "station_name": None})

# Combine into a single DataFrame
df_stations_ordered = pd.DataFrame(results)

# Keep only one row per native_id
df_stations_unique = df_stations_ordered.drop_duplicates(subset='native_id', keep='first')

# Then merge
df_merged = df_concat_new[['Station ID', 'old_name', 'new_name', 'new_native_id']].merge(
    df_stations_unique,
    left_on='new_native_id',
    right_on='native_id',
    how='left'
)

# Drop the duplicate column and rename
df_merged = df_merged.drop(columns=['native_id']).rename(columns={'station_name': 'db_station_name'})

df_merged

,Station ID,old_name,new_name,new_native_id,db_station_name
0,12939,,Fort St John Key Learning Centre,469,None
1,12938,,Pine River Gas Plant,107,Pine River Gas Plant
2,2640,Harmac Pacific,Harmac Pacific,294,MERRITT RS
3,2623,Kelowna College,Kelowna College,102,Kelowna College
4,12531,NaN,NaN,97,ZZ COPPER
5,12534,NaN,NaN,104,ZZ KEMANO
6,12535,NaN,NaN,439,Burns Lake Sheraton East Met_60
7,12537,NaN,NaN,553,Colwood City Hall
8,12539,NaN,NaN,526,ZZ SNUG
9,12540,NaN,NaN,551,Cranbrook Muriel Baxter


,Station ID,old_name,new_name,new_native_id,db_station_name
0,12939,,Fort St John Key Learning Centre,469,None
1,12938,,Pine River Gas Plant,107,Pine River Gas Plant
2,2640,Harmac Pacific,Harmac Pacific,294,MERRITT RS
3,2623,Kelowna College,Kelowna College,102,Kelowna College
4,12531,NaN,NaN,97,ZZ COPPER
5,12534,NaN,NaN,104,ZZ KEMANO
6,12535,NaN,NaN,439,Burns Lake Sheraton East Met_60
7,12537,NaN,NaN,553,Colwood City Hall
8,12539,NaN,NaN,526,ZZ SNUG
9,12540,NaN,NaN,551,Cranbrook Muriel Baxter


Horseshoe Bay -> T035, the attribution change